[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/36_moe_solution.ipynb)

# 🔴 Solution: Mixture of Experts (top-k routing)

*Attention & Transformers · Hard*

Reference implementation. Try it yourself in `36_moe.ipynb` first.

---
Implement a **top-k mixture-of-experts** layer as an `nnx.Module`, returning
both the output and the load-balancing auxiliary loss.

### Signature
```python
class MoELayer(nnx.Module):
    def __init__(self, d_model, d_hidden, num_experts, top_k=2, *, rngs):
        ...
    def __call__(self, x):
        ...  # (B, T, d_model) -> (output, aux_loss)
```

Each expert is an independent 2-layer MLP `d_model -> d_hidden -> d_model` with
a ReLU. The router is a single `(d_model, num_experts)` matrix.

### The routing
1. `logits = x @ W_router` → `(N, E)` for `N = B*T` tokens
2. `probs = softmax(logits)` over **all** experts — used by the aux loss
3. Select the top-$k$ experts per token
4. Renormalise **over the selected $k$ only**, so their weights sum to 1
5. Output is the weighted sum of those $k$ experts' outputs

### The auxiliary loss
$$\mathcal{L}_{\text{aux}} = E \sum_{e=1}^{E} f_e \cdot P_e$$

where $f_e$ is the **fraction of tokens** routed to expert $e$ (hard, from
top-$k$) and $P_e$ is the **mean router probability** for expert $e$ (soft, from
the full softmax). Perfectly uniform routing gives
$\mathcal{L}_{\text{aux}} = E \cdot E \cdot \frac{1}{E}\cdot\frac{1}{E} = 1$.

### Rules
- Renormalise over the selected experts only — this is the step people miss
- Return `(output, aux_loss)`; output shape matches the input
- The aux loss must be differentiable **through $P_e$** (the hard counts $f_e$
  are not differentiable, and that is fine — the gradient flows via $P$)

### Why the aux loss is not optional
Routing is a positive feedback loop: an expert that is slightly better early
gets more tokens, so it trains faster, so it gets picked more. Left alone this
collapses — a handful of experts take nearly all traffic and the rest are dead
weight, so you have paid for $E$ experts and are effectively running two.

The $f_e \cdot P_e$ product is a neat piece of design. $f$ is what you actually
care about but has no gradient (it comes from an argmax). $P$ is differentiable
but does not directly measure load. Multiplying them gives a term whose gradient
pushes down the router probability for experts that are *currently* overloaded,
with the load entering as a constant multiplier.

### Parameters vs FLOPs — the whole point
An MoE layer holds $E$ experts' worth of parameters but activates only $k$ per
token. With $E=64, k=2$ you get 32x the parameters at ~2 experts' compute.
Since capability scales with parameter count while cost scales with *active*
parameters, MoE buys capacity for cheap.

What it costs is memory and communication: all $E$ experts must be resident even
though most are idle for any given token, and at scale the experts are sharded
across devices so routing becomes an all-to-all — which is why real
implementations obsess over expert *capacity* and token dropping, and why the
naive dense-compute version below is correct but not fast.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class MoELayer(nnx.Module):
    """Top-k mixture of experts. Returns (output, aux_loss)."""

    def __init__(self, d_model: int, d_hidden: int, num_experts: int,
                 top_k: int = 2, *, rngs: nnx.Rngs):
        self.d_model = d_model
        self.num_experts = num_experts
        self.top_k = top_k

        key = rngs.params()
        k_router, k_w1, k_w2 = jax.random.split(key, 3)

        self.w_router = nnx.Param(
            jax.random.normal(k_router, (d_model, num_experts)) / jnp.sqrt(d_model)
        )
        # Experts stacked on a leading axis so they can be applied in one einsum.
        self.w1 = nnx.Param(
            jax.random.normal(k_w1, (num_experts, d_model, d_hidden)) / jnp.sqrt(d_model)
        )
        self.w2 = nnx.Param(
            jax.random.normal(k_w2, (num_experts, d_hidden, d_model)) / jnp.sqrt(d_hidden)
        )

    def __call__(self, x):
        B, T, D = x.shape
        E, k = self.num_experts, self.top_k
        flat = x.reshape(-1, D)                             # (N, D)
        N = flat.shape[0]

        logits = flat @ self.w_router.value                 # (N, E)
        probs = jax.nn.softmax(logits, axis=-1)             # full soft routing

        top_vals, top_idx = jax.lax.top_k(logits, k)        # (N, k)
        # Renormalise over the SELECTED experts only, so their weights sum to 1.
        top_w = jax.nn.softmax(top_vals, axis=-1)

        # Dense compute: every expert on every token, then gather. Correct and
        # simple; a production kernel would dispatch instead.
        h = jax.nn.relu(jnp.einsum("nd,edh->enh", flat, self.w1.value))
        all_out = jnp.einsum("enh,ehd->ned", h, self.w2.value)   # (N, E, D)

        # picked[n, j] = all_out[n, top_idx[n, j]]
        picked = jnp.take_along_axis(all_out, top_idx[:, :, None], axis=1)

        out = jnp.sum(picked * top_w[:, :, None], axis=1)    # (N, D)

        # Load balancing: hard fraction f_e times mean soft probability P_e.
        one_hot = jax.nn.one_hot(top_idx, E).sum(axis=1)     # (N, E) counts
        f = jnp.mean(one_hot, axis=0)                        # tokens per expert
        P = jnp.mean(probs, axis=0)                          # mean router prob
        aux_loss = E * jnp.sum(f * P)

        return out.reshape(B, T, D), aux_loss

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

layer = MoELayer(d_model=32, d_hidden=64, num_experts=8, top_k=2, rngs=nnx.Rngs(0))
x = jax.random.normal(jax.random.key(1), (2, 16, 32))

out, aux = layer(x)
print("out:", out.shape, " aux_loss:", float(aux))
print("(uniform routing would give aux_loss = 1.0)")

# How many parameters are there vs how many run per token?
p = nnx.state(layer, nnx.Param)
total = sum(v.size for v in jax.tree.leaves(p))
per_token = 32 * 64 * 2 * 2      # top_k experts, two matrices each
print(f"total expert params: {total}, active per token: ~{per_token}")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("moe")